In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_path     = "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_countries"
silver_path     = "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_countries"
quarantine_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_quarantine/netflix_countries"
checkpoint_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_checkpoints/netflix_countries"

def process_countries_microbatch(micro_df, batch_id):
    if micro_df.isEmpty():
        return

    if "_rescued_data" not in micro_df.columns:
        micro_df = micro_df.withColumn("_rescued_data", F.lit(None).cast("string"))

    silver_titles_ids = spark.read.format("delta").load(
        "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
    ).select("show_id").distinct()

    # 1. Strip commas/spaces, apply smart Title Case, and map short forms
    base_country = F.regexp_replace(
        F.regexp_replace(
            F.initcap(
                F.trim(
                    F.regexp_replace(
                        F.regexp_replace(F.col("country"), r"^[,\s]+|[,\s]+$", ""),
                        r"\s+",
                        " "
                    )
                )
            ),
            r"\bOf\b", "of"
        ),
        r"\bAnd\b", "and"
    )

    country_aliases = {
        "Usa": "United States",
        "Us": "United States",
        "U.s.": "United States",
        "U.s.a.": "United States",
        "United States of America": "United States",
        "Uk": "United Kingdom",
        "U.k.": "United Kingdom",
        "Great Britain": "United Kingdom",
        "England": "United Kingdom",
        "Uae": "United Arab Emirates",
        "U.a.e.": "United Arab Emirates"
    }
    mapping_expr = F.create_map([F.lit(x) for item in country_aliases.items() for x in item])

    staged = (
        micro_df
        .withColumn("show_id", F.trim(F.col("show_id").cast("string")))
        .withColumn(
            "country",
            F.when(F.col("country").isNull() | (F.trim(F.col("country")) == ""), F.lit(None).cast("string"))
             .otherwise(F.coalesce(mapping_expr[base_country], base_country))
        )
    )

    # 2. Check foreign-key match against Silver netflix_titles
    staged_with_fk = (
        staged.alias("c")
        .join(silver_titles_ids.alias("t"), F.col("c.show_id") == F.col("t.show_id"), "left")
        .select(
            F.col("c.*"),
            F.when(F.col("t.show_id").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("_is_orphan")
        )
    )

    # 3. Unified quarantine rules
    invalid_condition = (
        F.col("_rescued_data").isNotNull() |
        F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$") |
        F.col("country").isNull() | F.col("country").rlike(r"^[0-9\W_]+$") |
        F.col("_is_orphan")
    )

    quarantine_df = (
        staged_with_fk.filter(invalid_condition)
        .withColumn(
            "quarantine_reason",
            F.when(F.col("_rescued_data").isNotNull(), "Corrupt CSV / Rescued Data")
             .when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
             .when(F.col("country").isNull() | F.col("country").rlike(r"^[0-9\W_]+$"), "Invalid or Missing country")
             .when(F.col("_is_orphan"), "Orphan show_id (missing in silver.netflix_titles)")
             .otherwise("Unknown validation failure")
        )
        .drop("_is_orphan")
        .withColumn("_quarantined_at", F.current_timestamp())
    )

    clean_df = (
        staged_with_fk.filter(~invalid_condition)
        .drop("_is_orphan", "_rescued_data")
        .dropDuplicates(["show_id", "country"])
        .withColumn("_silver_processed_at", F.current_timestamp())
    )

    # 4. Write only new batch to Silver (Upsert) and Quarantine (Append)
    if not clean_df.isEmpty():
        if DeltaTable.isDeltaTable(spark, silver_path):
            (
                DeltaTable.forPath(spark, silver_path).alias("target")
                .merge(
                    clean_df.alias("source"),
                    "target.show_id = source.show_id AND target.country = source.country"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
        else:
            clean_df.write.format("delta").mode("overwrite").save(silver_path)

    if not quarantine_df.isEmpty():
        quarantine_df.write.format("delta").mode("append").option("mergeSchema", "true").save(quarantine_path)

query = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
    .writeStream
    .foreachBatch(process_countries_microbatch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print("✅ Incremental Silver processing complete for netflix_countries.")
